In [0]:
import base64, gzip, hashlib, json
dbutils.widgets.text("run_id", "")
dbutils.widgets.text("run_open_ts", "")
dbutils.widgets.text("source_update_id", "")
dbutils.widgets.text("silver_update_id", "")
dbutils.widgets.text("scratch_prefix", "")
RUN = dbutils.widgets.get("run_id").strip()
RUN_OPEN_TS = dbutils.widgets.get("run_open_ts").strip()
SOURCE_UPDATE_ID = dbutils.widgets.get("source_update_id").strip()
SILVER_UPDATE_ID = dbutils.widgets.get("silver_update_id").strip()
SCRATCH_PREFIX = dbutils.widgets.get("scratch_prefix").strip()
assert RUN.startswith("dq4_omop_") and RUN.replace("_", "").isalnum(), RUN
assert RUN_OPEN_TS and SOURCE_UPDATE_ID and SILVER_UPDATE_ID and SCRATCH_PREFIX
LANE="achilles"
payload="""..."""
rows=json.loads(gzip.decompress(base64.b64decode(payload)).decode())
TEMPLATE_REPLACEMENTS = {
    "dq4_omop_20260825_r5": RUN,
    "dq4_omop_20260825_r2": RUN,
    "2026-08-26T06:48:21.102Z": RUN_OPEN_TS,
    "cc682c9c-8795-4c48-adea-f988320f8d0d": SOURCE_UPDATE_ID,
    "f5c7c7ab-e37d-4a31-b9c2-b7631becb16a": SILVER_UPDATE_ID,
    "r5q9n3k6": SCRATCH_PREFIX,
}
def adapt(value):
    if isinstance(value, str):
        for old, new in TEMPLATE_REPLACEMENTS.items():
            value = value.replace(old, new)
        return value
    if isinstance(value, list):
        return [adapt(v) for v in value]
    if isinstance(value, dict):
        return {k: adapt(v) for k, v in value.items()}
    return value
rows = adapt(rows)
def q(v):
    return "'" + str(v).replace("'", "''") + "'"
def execute(seq,name,sql,missing_ok=False):
    sha=hashlib.sha256(sql.encode()).hexdigest()
    spark.sql(f"""INSERT INTO 8_dev.silver_qc.dq_exec_log
      (run_id,lane,stmt_file,seq,stmt_sha256,status,error_text,rows_affected,attempted_at,settled_at,created_by_session)
      VALUES ({q(RUN)},{q(LANE)},{q(name)},{seq},{q(sha)},'attempted',NULL,NULL,current_timestamp(),NULL,'DQ4')""")
    try:
        spark.sql(sql).collect()
        status="ok"; err="NULL"
    except Exception as e:
        msg=str(e)[:4000]
        if missing_ok and ("TABLE_OR_VIEW_NOT_FOUND" in msg or ("table or view" in msg.lower() and "cannot be found" in msg.lower())):
            status="skipped_missing_table"; err=q(msg)
        else:
            spark.sql(f"""INSERT INTO 8_dev.silver_qc.dq_exec_log
              (run_id,lane,stmt_file,seq,stmt_sha256,status,error_text,rows_affected,attempted_at,settled_at,created_by_session)
              VALUES ({q(RUN)},{q(LANE)},{q(name)},{seq},{q(sha)},'error',{q(msg)},NULL,current_timestamp(),current_timestamp(),'DQ4')""")
            raise
    spark.sql(f"""INSERT INTO 8_dev.silver_qc.dq_exec_log
      (run_id,lane,stmt_file,seq,stmt_sha256,status,error_text,rows_affected,attempted_at,settled_at,created_by_session)
      VALUES ({q(RUN)},{q(LANE)},{q(name)},{seq},{q(sha)},{q(status)},{err},NULL,current_timestamp(),current_timestamp(),'DQ4')""")
for row in rows:
    execute(row["seq"],row["path"],row["sql"],True)